In [ ]:
# Cell 1: Import required libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import re
import urllib.request
import zipfile
import os

In [ ]:
# Cell 2: Download dataset from Kaggle URL (Shakespeare Sonnets)
# Using a public domain Shakespeare sonnets dataset from Kaggle
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = urllib.request.urlopen(url)
data = response.read().decode('utf-8')

print(f"Dataset downloaded. Length: {len(data)} characters")
print(f"First 500 characters:\n{data[:500]}")

In [ ]:
# Cell 3: Text preprocessing
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^a-zA-Z\s\.\,\!\?\']', '', text)
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text)
    # Remove leading/trailing spaces
    text = text.strip()
    return text

# Clean the entire text
cleaned_text = clean_text(data)
print(f"Original length: {len(data)}")
print(f"Cleaned length: {len(cleaned_text)}")
print(f"\nSample cleaned text:\n{cleaned_text[:500]}")

In [ ]:
# Cell 4: Prepare sequences for training
# Split text into sentences/lines
lines = cleaned_text.split('\n')
lines = [line.strip() for line in lines if len(line.strip()) > 20]

print(f"Number of lines: {len(lines)}")
print(f"Sample lines:\n{lines[:5]}")

In [ ]:
# Cell 5: Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)
total_words = len(tokenizer.word_index) + 1

print(f"Total unique words: {total_words}")
print(f"First 10 words in vocabulary: {list(tokenizer.word_index.items())[:10]}")

In [ ]:
# Cell 6: Create input sequences
input_sequences = []

for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total input sequences created: {len(input_sequences)}")
print(f"Sample sequence: {input_sequences[0]}")

In [ ]:
# Cell 7: Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
print(f"Maximum sequence length: {max_sequence_len}")

# Pad sequences
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
print(f"Padded sequences shape: {padded_sequences.shape}")

In [ ]:
# Cell 8: Create features and labels
X = padded_sequences[:, :-1]  # All words except last
y = padded_sequences[:, -1]   # Last word

# Convert labels to categorical
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Cell 9: Build LSTM model
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len-1),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dropout(0.2),
    Dense(total_words, activation='softmax')
])

# Compile model
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Cell 10: Train model
history = model.fit(
    X, y,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Cell 11: Save model and tokenizer
# Save model
model.save('textgen_model.h5')
print("Model saved as 'textgen_model.h5'")

# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Tokenizer saved as 'tokenizer.pkl'")